In [ ]:
import json
import os
import subprocess
# Repo + global constants -----------------------------------------------------
repo_root = subprocess.check_output(["git", "rev-parse", "--show-toplevel"]).decode().strip()
os.chdir(repo_root)
print("cwd =", os.getcwd())
from pathlib import Path

import numpy as np
import polars as pl
import torch
from itables import show as itshow
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
from torch.utils.data import DataLoader, TensorDataset, random_split

from models.VAE.default import UNetVAE
from train.VAE.train_VAE_unet import load_sigma_dataset, kl_divergence
from train.util import TVHuberLoss3D


REPO_ROOT = Path(repo_root)
RUNS_ROOT = REPO_ROOT / "saved_runs" / "grid_search_vae_2"
LOG_EPS = 1e-6
SPLIT_SEED = 159753
TV_LAMBDA = 3e-4
RECON_BETA = 0.5

DROP_COLS = {"run_dir", "beta", "kl_warmup_epochs", "batch_size", "epochs",
             "best_checkpoint", "final_checkpoint", "training_done"}

# Helper functions ------------------------------------------------------------
def find_checkpoint(run_dir: Path, names: tuple[str, ...]) -> Path | None:
    for name in names:
        path = run_dir / "checkpoints" / name
        if path.exists():
            return path
    return None


def build_test_loader(data_path: Path, sparse: bool, batch_size: int, device: torch.device) -> DataLoader:
    sigma, _, _, _ = load_sigma_dataset(data_path, log_eps=LOG_EPS, sparse=sparse)
    sigma = sigma.to(device)
    dataset = TensorDataset(sigma)
    n = len(dataset)
    train_size = int(0.8 * n)
    val_size = int(0.1 * n)
    test_size = n - train_size - val_size
    _, _, test_ds = random_split(
        dataset,
        [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(SPLIT_SEED),
    )
    return DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)


def load_vae(ckpt_path: Path, bc: int, lc: int, device: torch.device) -> UNetVAE:
    model = UNetVAE(base_channels=bc, latent_channels=lc).to(device)
    checkpoint = torch.load(ckpt_path, map_location="cpu")
    ema_state = checkpoint.get("ema_state_dict")
    if ema_state is not None:
        ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.999))
        ema.load_state_dict(ema_state)
        model.load_state_dict(ema.module.state_dict())
    else:
        model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    return model


def relative_metrics(pred: torch.Tensor, target: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    pred_flat = pred.float().reshape(pred.shape[0], -1)
    target_flat = target.float().reshape(target.shape[0], -1)
    diff = pred_flat - target_flat
    rel_l2 = torch.linalg.vector_norm(diff, ord=2, dim=1) / torch.clamp(
        torch.linalg.vector_norm(target_flat, ord=2, dim=1), min=1e-12
    )
    rel_l1 = torch.linalg.vector_norm(diff, ord=1, dim=1) / torch.clamp(
        torch.linalg.vector_norm(target_flat, ord=1, dim=1), min=1e-12
    )
    return rel_l1, rel_l2


def summarize(values: list[float], prefix: str) -> dict[str, float | None]:
    if not values:
        return {
            f"mean_{prefix}": None,
            f"median_{prefix}": None,
            f"min_{prefix}": None,
            f"max_{prefix}": None,
        }
    arr = np.asarray(values, dtype=np.float64)
    return {
        f"mean_{prefix}": float(arr.mean()),
        f"median_{prefix}": float(np.median(arr)),
        f"min_{prefix}": float(arr.min()),
        f"max_{prefix}": float(arr.max()),
    }


def evaluate_run(run_dir: Path, device: torch.device) -> dict[str, object]:
    hparams_path = run_dir / "hparams.json"
    with hparams_path.open("r", encoding="utf-8") as f:
        hparams = json.load(f)

    sparse = str(hparams.get("sparse", "false")).strip().lower() == "true"
    bc = int(hparams["bc"])
    lc = int(hparams["lc"])
    batch_size = int(hparams["batch_size"])
    beta = float(hparams["beta"])
    kl_warmup_epochs = int(hparams["kl_warmup_epochs"])

    best_ckpt = find_checkpoint(run_dir, ("best_val_loss.pt",))
    final_ckpt = find_checkpoint(run_dir, ("final_model.pt",))

    row: dict[str, object] = {
        "run_dir": str(run_dir),
        "save_dir": hparams.get("save_dir", run_dir.name),
        "sparse": sparse,
        "bc": bc,
        "lc": lc,
        "lr": float(hparams["lr"]),
        "beta": beta,
        "kl_warmup_epochs": kl_warmup_epochs,
        "batch_size": batch_size,
        "epochs": int(hparams["epochs"]),
        "best_checkpoint": str(best_ckpt) if best_ckpt is not None else None,
        "final_checkpoint": str(final_ckpt) if final_ckpt is not None else None,
        "training_done": best_ckpt is not None and final_ckpt is not None,
        "status": "ok" if best_ckpt is not None and final_ckpt is not None else "training_not_done",
    }
    for prefix in ("recon", "kl", "total", "rel_l1", "rel_l2"):
        row.update(summarize([], prefix))

    if best_ckpt is None or final_ckpt is None:
        return row

    data_path = REPO_ROOT / "data" / "3dert_test2.npy"
    test_loader = build_test_loader(data_path, sparse=sparse, batch_size=batch_size, device=device)

    checkpoint = torch.load(best_ckpt, map_location="cpu")
    model = load_vae(best_ckpt, bc=bc, lc=lc, device=device)
    epoch = int(checkpoint.get("epoch", kl_warmup_epochs))
    kl_weight = beta * min(1.0, epoch / max(1, kl_warmup_epochs))

    loss_fn = TVHuberLoss3D(lam_tv=TV_LAMBDA, beta=RECON_BETA, w_in=10.0, thresh=0.5)

    recon_values: list[float] = []
    kl_values: list[float] = []
    total_values: list[float] = []
    rel_l1_values: list[float] = []
    rel_l2_values: list[float] = []

    with torch.no_grad():
        for (x,) in test_loader:
            x = x.to(device)
            recon, mu, logvar = model(x)
            recon_loss = loss_fn(recon, x)
            kl_loss = kl_divergence(mu, logvar)
            total_loss = recon_loss + kl_weight * kl_loss
            rel_l1, rel_l2 = relative_metrics(recon, x)

            n = x.shape[0]
            recon_values.extend([recon_loss.item()] * n)
            kl_values.extend([kl_loss.item()] * n)
            total_values.extend([total_loss.item()] * n)
            rel_l1_values.extend(rel_l1.detach().cpu().tolist())
            rel_l2_values.extend(rel_l2.detach().cpu().tolist())

    row.update(summarize(recon_values, "recon"))
    row.update(summarize(kl_values, "kl"))
    row.update(summarize(total_values, "total"))
    row.update(summarize(rel_l1_values, "rel_l1"))
    row.update(summarize(rel_l2_values, "rel_l2"))
    return row

# Evaluation flow -------------------------------------------------------------
device = torch.device("cuda:0")
run_dirs = sorted(path.parent for path in RUNS_ROOT.rglob("hparams.json"))

vae_run_dirs = [
    d
    for d in run_dirs
    if "bc" in json.loads((d / "hparams.json").read_text())
    and "unet_ch" not in json.loads((d / "hparams.json").read_text())
]
print(f"Found {len(vae_run_dirs)} VAE run(s) under {RUNS_ROOT}")

rows = [evaluate_run(run_dir, device) for run_dir in vae_run_dirs]
if not rows:
    raise SystemExit("No VAE runs found.")

df = pl.DataFrame(rows).sort(
    by=["status", "mean_rel_l2", "save_dir"],
    descending=[False, False, False],
    nulls_last=True,
)

display_df = df.drop([c for c in DROP_COLS if c in df.columns]).to_pandas()
itshow(display_df, classes="display compact", paging=False)

completed_df = df.filter(pl.col("status") == "ok")
metrics_to_report = (
    ("mean_rel_l2", "Relative L2"),
    ("mean_recon", "Reconstruction"),
    ("mean_kl", "KL"),
)

best_entries: list[tuple[str, str, str, dict[str, object]]] = []
worst_entries: list[tuple[str, str, str, dict[str, object]]] = []

for sparse_value, dataset_label in ((True, "Sparse"), (False, "Dense")):
    dataset_df = completed_df.filter(pl.col("sparse") == sparse_value)
    if dataset_df.height == 0:
        continue
    for metric_col, metric_label in metrics_to_report:
        metric_df = dataset_df.filter(pl.col(metric_col).is_not_null())
        if metric_df.height == 0:
            continue
        best_row = metric_df.sort(metric_col).row(0, named=True)
        worst_row = metric_df.sort(metric_col, descending=True).row(0, named=True)
        best_entries.append((dataset_label, metric_label, metric_col, best_row))
        worst_entries.append((dataset_label, metric_label, metric_col, worst_row))


COLOR_RED = "\033[31m"
COLOR_PURPLE = "\033[35m"
COLOR_RESET = "\033[0m"
def print_entries(entries, label, color):
    counts = {}
    for _, _, _, row in entries:
        counts[row["save_dir"]] = counts.get(row["save_dir"], 0) + 1

    for dataset_label, metric_label, metric_col, row in entries:
        name = row["save_dir"]
        value = row[metric_col]
        name_display = (
            f"{color}{name}{COLOR_RESET}" if counts[name] > 1 else name
        )
        prefix = "best" if label == "best" else "worst"
        print(
            f"{dataset_label} {prefix} ({metric_label}): {name_display} "
            f"({metric_col}={value:.6f})"
        )

print_entries(best_entries, "best", COLOR_RED)
print_entries(worst_entries, "worst", COLOR_PURPLE)

cwd = /home/johnma/ert3d_test2
Found 7 VAE run(s) under /home/johnma/ert3d_test2/saved_runs/grid_search_vae_2


Loading ITables v2.7.1 from the internet... (need help?)


Sparse best (Relative L2): grid_search_vae_2/_lr2e-04_beta5e-02_kl400_bs128_ep1200_bc8_lc4_sparse (mean_rel_l2=0.039665)
Sparse best (Reconstruction): grid_search_vae_2/_lr2e-04_beta5e-02_kl400_bs128_ep1200_bc8_lc4_sparse (mean_recon=0.874316)
Sparse best (KL): grid_search_vae_2/_lr2e-04_beta5e-02_kl400_bs128_ep1200_bc16_lc4_sparse (mean_kl=0.346149)
Dense best (Relative L2): grid_search_vae_2/_lr2e-04_beta5e-02_kl400_bs128_ep1200_bc8_lc4 (mean_rel_l2=0.042947)
Dense best (Reconstruction): grid_search_vae_2/_lr2e-04_beta5e-02_kl400_bs128_ep1200_bc8_lc4 (mean_recon=0.982719)
Dense best (KL): grid_search_vae_2/_lr2e-04_beta5e-02_kl400_bs128_ep1200_bc8_lc8 (mean_kl=0.352270)
Sparse worst (Relative L2): grid_search_vae_2/_lr2e-04_beta5e-02_kl400_bs128_ep1200_bc16_lc4_sparse (mean_rel_l2=0.077872)
Sparse worst (Reconstruction): grid_search_vae_2/_lr2e-04_beta5e-02_kl400_bs128_ep1200_bc16_lc4_sparse (mean_recon=2.422344)
Sparse worst (KL): grid_search_vae_2/_lr2e-04_beta5e-02_kl400_bs128_ep1

: 